# Brooklyn Residential Sales – Data Preprocessing

## Objective
Prepare a clean, structured dataset of Brooklyn residential property sales suitable for feature engineering and predictive modeling.

This notebook focuses strictly on:
- Data cleaning
- Filtering to relevant property types
- Handling missing values
- Removing anomalous transactions
- Preparing consistent identifiers for downstream merging

No modeling is performed in this notebook.

##### Import Libraries

In [1]:
import pandas as pd
import numpy as np
import seaborn as sns
import openpyxl
sns.set()

## Data Sources

- NYC Residential Sales dataset
- PLUTO property dataset (for geographic attributes)
- MTA Subway Stations dataset (for transit proximity)

All datasets are merged downstream in feature engineering.

##### Real Estate Data

In [4]:
# File path locations
from pathlib import Path

# File paths
PROJECT_ROOT = Path.cwd().resolve().parent
CLEAN_DATA_DIR = PROJECT_ROOT / "data/clean_data"
RAW_DATA_DIR = PROJECT_ROOT / "data/raw_data"
MODELS_DIR = PROJECT_ROOT / "models"
SRC_DIR = PROJECT_ROOT / "src"
DEPLOYMENT_DIR = PROJECT_ROOT / "deployment"

In [5]:
data = pd.read_excel(RAW_DATA_DIR / "rollingsales_brooklyn.xlsx", sheet_name="Brooklyn")
pd.set_option('display.max_columns', None)
data.head()

,BOROUGH,NEIGHBORHOOD,BUILDING CLASS CATEGORY,TAX CLASS AT PRESENT,BLOCK,LOT,EASEMENT,BUILDING CLASS AT PRESENT,ADDRESS,APARTMENT NUMBER,ZIP CODE,RESIDENTIAL UNITS,COMMERCIAL UNITS,TOTAL UNITS,LAND SQUARE FEET,GROSS SQUARE FEET,YEAR BUILT,TAX CLASS AT TIME OF SALE,BUILDING CLASS AT TIME OF SALE,SALE PRICE,SALE DATE
0,3,BATH BEACH,01 ONE FAMILY DWELLINGS,1,6360,63,NaN,A5,48 BAY 10 STREET,NaN,11228,1.0,0.0,1.0,1547.0,1428.0,1930.0,1,A5,0,2025-02-03
1,3,BATH BEACH,01 ONE FAMILY DWELLINGS,1,6364,2,NaN,A5,1649 BENSON AVENUE,NaN,11214,1.0,0.0,1.0,1638.0,972.0,1930.0,1,A5,0,2025-04-08
2,3,BATH BEACH,01 ONE FAMILY DWELLINGS,1,6364,4,NaN,A5,1643 BENSON AVENUE,NaN,11214,1.0,0.0,1.0,1638.0,1746.0,1930.0,1,A5,0,2025-09-08
3,3,BATH BEACH,01 ONE FAMILY DWELLINGS,1,6364,72,NaN,A5,68 BAY 14TH STREET,NaN,11214,1.0,0.0,1.0,1950.0,972.0,1950.0,1,A5,860000,2025-11-24
4,3,BATH BEACH,01 ONE FAMILY DWELLINGS,1,6372,39,NaN,A9,86-29 19 AVENUE,NaN,11214,1.0,0.0,1.0,2417.0,1944.0,1930.0,1,A9,1180000,2025-11-19


In [6]:
data.shape

(21857, 21)

In [7]:
# Standardize column names
data.columns = (
    data.columns.str.strip()
             .str.lower()
             .str.replace(" ", "_")
             .str.replace("-", "_")
)

In [8]:
# Columns to make into INT
cols_to_int = ["residential_units", "commercial_units", "total_units", 
               "land_square_feet", "gross_square_feet", "year_built"]

for c in cols_to_int:
    data[c] = data[c].fillna(0).astype("int64")

data.head()

,borough,neighborhood,building_class_category,tax_class_at_present,block,lot,easement,building_class_at_present,address,apartment_number,zip_code,residential_units,commercial_units,total_units,land_square_feet,gross_square_feet,year_built,tax_class_at_time_of_sale,building_class_at_time_of_sale,sale_price,sale_date
0,3,BATH BEACH,01 ONE FAMILY DWELLINGS,1,6360,63,NaN,A5,48 BAY 10 STREET,NaN,11228,1,0,1,1547,1428,1930,1,A5,0,2025-02-03
1,3,BATH BEACH,01 ONE FAMILY DWELLINGS,1,6364,2,NaN,A5,1649 BENSON AVENUE,NaN,11214,1,0,1,1638,972,1930,1,A5,0,2025-04-08
2,3,BATH BEACH,01 ONE FAMILY DWELLINGS,1,6364,4,NaN,A5,1643 BENSON AVENUE,NaN,11214,1,0,1,1638,1746,1930,1,A5,0,2025-09-08
3,3,BATH BEACH,01 ONE FAMILY DWELLINGS,1,6364,72,NaN,A5,68 BAY 14TH STREET,NaN,11214,1,0,1,1950,972,1950,1,A5,860000,2025-11-24
4,3,BATH BEACH,01 ONE FAMILY DWELLINGS,1,6372,39,NaN,A9,86-29 19 AVENUE,NaN,11214,1,0,1,2417,1944,1930,1,A9,1180000,2025-11-19


In [9]:
types_null_counts = pd.DataFrame()
types_null_counts["Data Type"] = data.dtypes
types_null_counts["Count of Nulls"] = data.isna().sum()
types_null_counts

,Data Type,Count of Nulls
borough,int64,0
neighborhood,object,0
building_class_category,object,0
tax_class_at_present,object,0
block,int64,0
lot,int64,0
easement,float64,21857
building_class_at_present,object,0
address,object,0
apartment_number,object,16283


In [10]:
data.head()

,borough,neighborhood,building_class_category,tax_class_at_present,block,lot,easement,building_class_at_present,address,apartment_number,zip_code,residential_units,commercial_units,total_units,land_square_feet,gross_square_feet,year_built,tax_class_at_time_of_sale,building_class_at_time_of_sale,sale_price,sale_date
0,3,BATH BEACH,01 ONE FAMILY DWELLINGS,1,6360,63,NaN,A5,48 BAY 10 STREET,NaN,11228,1,0,1,1547,1428,1930,1,A5,0,2025-02-03
1,3,BATH BEACH,01 ONE FAMILY DWELLINGS,1,6364,2,NaN,A5,1649 BENSON AVENUE,NaN,11214,1,0,1,1638,972,1930,1,A5,0,2025-04-08
2,3,BATH BEACH,01 ONE FAMILY DWELLINGS,1,6364,4,NaN,A5,1643 BENSON AVENUE,NaN,11214,1,0,1,1638,1746,1930,1,A5,0,2025-09-08
3,3,BATH BEACH,01 ONE FAMILY DWELLINGS,1,6364,72,NaN,A5,68 BAY 14TH STREET,NaN,11214,1,0,1,1950,972,1950,1,A5,860000,2025-11-24
4,3,BATH BEACH,01 ONE FAMILY DWELLINGS,1,6372,39,NaN,A9,86-29 19 AVENUE,NaN,11214,1,0,1,2417,1944,1930,1,A9,1180000,2025-11-19


In [11]:
data["build_age_yrs"] = 2026 - data["year_built"]
data.head()

,borough,neighborhood,building_class_category,tax_class_at_present,block,lot,easement,building_class_at_present,address,apartment_number,zip_code,residential_units,commercial_units,total_units,land_square_feet,gross_square_feet,year_built,tax_class_at_time_of_sale,building_class_at_time_of_sale,sale_price,sale_date,build_age_yrs
0,3,BATH BEACH,01 ONE FAMILY DWELLINGS,1,6360,63,NaN,A5,48 BAY 10 STREET,NaN,11228,1,0,1,1547,1428,1930,1,A5,0,2025-02-03,96
1,3,BATH BEACH,01 ONE FAMILY DWELLINGS,1,6364,2,NaN,A5,1649 BENSON AVENUE,NaN,11214,1,0,1,1638,972,1930,1,A5,0,2025-04-08,96
2,3,BATH BEACH,01 ONE FAMILY DWELLINGS,1,6364,4,NaN,A5,1643 BENSON AVENUE,NaN,11214,1,0,1,1638,1746,1930,1,A5,0,2025-09-08,96
3,3,BATH BEACH,01 ONE FAMILY DWELLINGS,1,6364,72,NaN,A5,68 BAY 14TH STREET,NaN,11214,1,0,1,1950,972,1950,1,A5,860000,2025-11-24,76
4,3,BATH BEACH,01 ONE FAMILY DWELLINGS,1,6372,39,NaN,A9,86-29 19 AVENUE,NaN,11214,1,0,1,2417,1944,1930,1,A9,1180000,2025-11-19,96


In [12]:
# Create borough-block-lot as one column
data["BBL"] = (
    data["borough"].astype(str) + 
    data["block"].astype(str).str.zfill(5) + 
    data["lot"].astype(str).str.zfill(4)
).astype("int64")

data.head()

,borough,neighborhood,building_class_category,tax_class_at_present,block,lot,easement,building_class_at_present,address,apartment_number,zip_code,residential_units,commercial_units,total_units,land_square_feet,gross_square_feet,year_built,tax_class_at_time_of_sale,building_class_at_time_of_sale,sale_price,sale_date,build_age_yrs,BBL
0,3,BATH BEACH,01 ONE FAMILY DWELLINGS,1,6360,63,NaN,A5,48 BAY 10 STREET,NaN,11228,1,0,1,1547,1428,1930,1,A5,0,2025-02-03,96,3063600063
1,3,BATH BEACH,01 ONE FAMILY DWELLINGS,1,6364,2,NaN,A5,1649 BENSON AVENUE,NaN,11214,1,0,1,1638,972,1930,1,A5,0,2025-04-08,96,3063640002
2,3,BATH BEACH,01 ONE FAMILY DWELLINGS,1,6364,4,NaN,A5,1643 BENSON AVENUE,NaN,11214,1,0,1,1638,1746,1930,1,A5,0,2025-09-08,96,3063640004
3,3,BATH BEACH,01 ONE FAMILY DWELLINGS,1,6364,72,NaN,A5,68 BAY 14TH STREET,NaN,11214,1,0,1,1950,972,1950,1,A5,860000,2025-11-24,76,3063640072
4,3,BATH BEACH,01 ONE FAMILY DWELLINGS,1,6372,39,NaN,A9,86-29 19 AVENUE,NaN,11214,1,0,1,2417,1944,1930,1,A9,1180000,2025-11-19,96,3063720039


##### Filter to Residential Sales

We restrict the dataset to residential property classes (houses, condos, apartments).

This ensures:
- Modeling relevance
- Elimination of commercial and mixed-use distortions
- Cleaner price signal

In [13]:
resid_only = data[(data["building_class_at_present"].str.contains("A|B|C|D")) & (data["sale_price"] > 0)]

resid_only.head()

,borough,neighborhood,building_class_category,tax_class_at_present,block,lot,easement,building_class_at_present,address,apartment_number,zip_code,residential_units,commercial_units,total_units,land_square_feet,gross_square_feet,year_built,tax_class_at_time_of_sale,building_class_at_time_of_sale,sale_price,sale_date,build_age_yrs,BBL
3,3,BATH BEACH,01 ONE FAMILY DWELLINGS,1,6364,72,NaN,A5,68 BAY 14TH STREET,NaN,11214,1,0,1,1950,972,1950,1,A5,860000,2025-11-24,76,3063640072
4,3,BATH BEACH,01 ONE FAMILY DWELLINGS,1,6372,39,NaN,A9,86-29 19 AVENUE,NaN,11214,1,0,1,2417,1944,1930,1,A9,1180000,2025-11-19,96,3063720039
9,3,BATH BEACH,01 ONE FAMILY DWELLINGS,1,6401,46,NaN,A1,144 BAY 17 STREET,NaN,11214,1,0,1,4833,2164,1925,1,A1,1300000,2025-01-13,101,3064010046
13,3,BATH BEACH,01 ONE FAMILY DWELLINGS,1,6431,15,NaN,A9,221 BAY 13 STREET,NaN,11214,1,0,1,1530,2484,1920,1,A9,900000,2025-10-20,106,3064310015
14,3,BATH BEACH,01 ONE FAMILY DWELLINGS,1,6431,56,NaN,A5,200 BAY 14 STREET,NaN,11214,1,0,1,2086,1458,1910,1,A5,930000,2025-02-05,116,3064310056


In [14]:
resid_sales_bk = resid_only.drop(["easement"], axis=1)

#### MapPLUTO Dataset

In [16]:
pluto = pd.read_csv(RAW_DATA_DIR / "Primary_Land_Use_Tax_Lot_Output_(PLUTO)_20260218.csv", 
                    low_memory=False)
pluto.head()

,borough,Tax block,Tax lot,community board,census tract 2010,cb2010,schooldist,council district,postcode,firecomp,policeprct,healtharea,sanitboro,sanitsub,address,zonedist1,zonedist2,zonedist3,zonedist4,overlay1,overlay2,spdist1,spdist2,spdist3,ltdheight,splitzone,bldgclass,landuse,easements,ownertype,ownername,lotarea,bldgarea,comarea,resarea,officearea,retailarea,garagearea,strgearea,factryarea,otherarea,areasource,numbldgs,numfloors,unitsres,unitstotal,lotfront,lotdepth,bldgfront,bldgdepth,ext,proxcode,irrlotcode,lottype,bsmtcode,assessland,assesstot,exempttot,yearbuilt,yearalter1,yearalter2,histdist,landmark,builtfar,residfar,commfar,facilfar,borocode,BBL,condono,tract2010,xcoord,ycoord,latitude,longitude,zonemap,zmcode,sanborn,taxmap,edesignum,appbbl,appdate,plutomapid,version,sanitdistrict,healthcenterdistrict,firm07_flag,pfirm15_flag,dcpedited,notes,bct2020,bctcb2020,geom,basempdate,dcasdate,edesigdate,landmkdate,masdate,polidate,rpaddate,zoningdate
0,QN,11015,1,412.0,522.0,1011.0,29.0,27.0,11412.0,E317,113.0,3520.0,4.0,6E,114-03 198 STREET,R4-1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,False,A5,1.0,0.0,NaN,"SANKER, BERNICE","3,052","1,924",0,"1,924",0,0,0,0,0,0,2.0,1,2.0,1,1,61.03,50,26,37,N,1.0,False,3.0,2.0,13200.0,55200.0,31680.0,1945.0,0,0,NaN,NaN,0.63,1.00,0.0,2.0,4,4110150001,NaN,522.0,1051854.0,194191.0,40.699428,-73.756191,15b,NaN,407 076,44804.0,NaN,4.110150e+09,08/16/2013,1,25v3.1,12.0,44.0,NaN,NaN,NaN,NaN,4052200.0,4.052200e+10,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,QN,10992,35,412.0,522.0,1007.0,29.0,27.0,11412.0,L150,113.0,3520.0,4.0,4E,113-41 197 STREET,R2,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,False,B3,1.0,0.0,NaN,"SINGH, HARDAI","3,000","1,672",0,"1,672",0,0,0,0,0,0,2.0,2,2.5,2,2,30,100,20,28,EG,1.0,False,5.0,2.0,12720.0,40920.0,0.0,1925.0,0,0,NaN,NaN,0.56,0.75,0.0,1.0,4,4109920035,NaN,522.0,1051580.0,194247.0,40.699584,-73.757179,15b,NaN,407 076,44804.0,NaN,NaN,NaN,1,25v3.1,12.0,44.0,NaN,NaN,NaN,NaN,4052200.0,4.052200e+10,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,QN,11014,22,412.0,522.0,1010.0,29.0,27.0,11412.0,E317,113.0,3520.0,4.0,6E,114-34 198 STREET,R4B,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,False,A5,1.0,0.0,NaN,"ROSEBOROUGH, SHERRIE","2,000","1,160",0,"1,160",0,0,0,0,0,0,2.0,1,2.0,1,1,20,100,20,30,N,3.0,False,5.0,2.0,9240.0,36900.0,1420.0,1945.0,0,0,NaN,NaN,0.58,1.00,0.0,2.0,4,4110140022,NaN,522.0,1051823.0,193855.0,40.698506,-73.756306,15b,NaN,407 076,44804.0,NaN,NaN,NaN,1,25v3.1,12.0,44.0,NaN,NaN,NaN,NaN,4052200.0,4.052200e+10,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,QN,11014,59,412.0,522.0,1010.0,29.0,27.0,11412.0,E317,113.0,3520.0,4.0,6E,114-39 197 STREET,R4B,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,False,A5,1.0,0.0,NaN,"SWEETWINE, SARAH G","2,500",870,0,870,0,0,0,0,0,0,2.0,1,2.0,1,1,25,100,16,28,N,2.0,False,5.0,2.0,9420.0,32460.0,20310.0,1950.0,0,0,NaN,NaN,0.35,1.00,0.0,2.0,4,4110140059,NaN,522.0,1051737.0,193797.0,40.698347,-73.756617,15b,NaN,407 076,44804.0,NaN,NaN,NaN,1,25v3.1,12.0,44.0,NaN,NaN,NaN,NaN,4052200.0,4.052200e+10,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,QN,10992,24,412.0,522.0,1007.0,29.0,27.0,11412.0,L150,113.0,3520.0,4.0,4E,113-26 198 STREET,R2,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,False,A1,1.0,0.0,NaN,"KEANE, LLOYD","3,000","1,672",0,"1,672",0,0,0,0,0,0,2.0,2,2.5,1,1,30,100,20,28,G,1.0,False,5.0,2.0,12840.0,39780.0,0.0,1930.0,0,0,NaN,NaN,0.56,0.75,0.0,1.0,4,4109920024,NaN,522.0,1051631.0,194410.0,40.700031,-73.756993,15b,NaN,407 076,44804.0,NaN,NaN,NaN,1,25v3.1,12.0,44.0,NaN,NaN,NaN,NaN,4052200.0,4.052200e+10,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [17]:
pluto_refined_bk = pluto.loc[pluto["borough"] == "BK", ["borough", "BBL", "latitude", "longitude"]]
pluto_refined_bk.head()

,borough,BBL,latitude,longitude
10,BK,3088960016,40.592717,-73.929812
11,BK,3088960044,40.592204,-73.929885
12,BK,3089390096,40.593601,-73.929534
13,BK,3088960042,40.592130,-73.929806
14,BK,3089460999,40.594416,-73.929796


In [18]:
resid_with_geo = resid_sales_bk.merge(
    pluto_refined_bk, 
    how="inner", 
    on="BBL"
    )

resid_with_geo.head()

,borough_x,neighborhood,building_class_category,tax_class_at_present,block,lot,building_class_at_present,address,apartment_number,zip_code,residential_units,commercial_units,total_units,land_square_feet,gross_square_feet,year_built,tax_class_at_time_of_sale,building_class_at_time_of_sale,sale_price,sale_date,build_age_yrs,BBL,borough_y,latitude,longitude
0,3,BATH BEACH,01 ONE FAMILY DWELLINGS,1,6364,72,A5,68 BAY 14TH STREET,NaN,11214,1,0,1,1950,972,1950,1,A5,860000,2025-11-24,76,3063640072,BK,40.607931,-74.007289
1,3,BATH BEACH,01 ONE FAMILY DWELLINGS,1,6372,39,A9,86-29 19 AVENUE,NaN,11214,1,0,1,2417,1944,1930,1,A9,1180000,2025-11-19,96,3063720039,BK,40.605178,-74.000846
2,3,BATH BEACH,01 ONE FAMILY DWELLINGS,1,6401,46,A1,144 BAY 17 STREET,NaN,11214,1,0,1,4833,2164,1925,1,A1,1300000,2025-01-13,101,3064010046,BK,40.604878,-74.006706
3,3,BATH BEACH,01 ONE FAMILY DWELLINGS,1,6431,15,A9,221 BAY 13 STREET,NaN,11214,1,0,1,1530,2484,1920,1,A9,900000,2025-10-20,106,3064310015,BK,40.605037,-74.010865
4,3,BATH BEACH,01 ONE FAMILY DWELLINGS,1,6431,56,A5,200 BAY 14 STREET,NaN,11214,1,0,1,2086,1458,1910,1,A5,930000,2025-02-05,116,3064310056,BK,40.605227,-74.010127


##### Filter to Residential Sales

We restrict the dataset to residential property classes (houses, condos, apartments).

This ensures:
- Modeling relevance
- Elimination of commercial and mixed-use distortions
- Cleaner price signal

In [19]:
resid_with_geo_no_na = resid_with_geo.dropna(subset=["latitude", "longitude"]).copy()

In [20]:
resid_with_geo_no_na.shape

(8229, 25)

#### MTA Station Data

In [22]:
# MTA Subway stations
mta = pd.read_csv(RAW_DATA_DIR / "MTA_Subway_Stations.csv")
mta.head()

,GTFS Stop ID,Station ID,Complex ID,Division,Line,Stop Name,Borough,CBD,Daytime Routes,Structure,GTFS Latitude,GTFS Longitude,North Direction Label,South Direction Label,ADA,ADA Northbound,ADA Southbound,ADA Notes,Georeference
0,127,317,611,IRT,Broadway - 7Av,Times Sq-42 St,M,True,1 2 3,Subway,40.755290,-73.987495,Uptown,Downtown,1,1,1,NaN,POINT (-73.987495 40.75529)
1,S17,515,515,SIR,Staten Island,Annadale,SI,False,SIR,Open Cut,40.540460,-74.178217,Ferry,South Shore,0,0,0,NaN,POINT (-74.178217 40.54046)
2,S01,139,627,BMT,Franklin Shuttle,Franklin Av,Bk,False,S,Elevated,40.680596,-73.955827,Last Stop,Prospect Park,1,1,1,NaN,POINT (-73.955827 40.680596)
3,254,349,349,IRT,Eastern Pky,Junius St,Bk,False,3,Elevated,40.663515,-73.902447,Manhattan,New Lots,0,0,0,NaN,POINT (-73.902447 40.663515)
4,M01,108,108,BMT,Myrtle Av,Middle Village-Metropolitan Av,Q,False,M,Elevated,40.711396,-73.889601,Inbound,Last Stop,1,1,1,NaN,POINT (-73.889601 40.711396)


In [23]:
mta.columns = (
    mta.columns.str.strip()
             .str.lower()
             .str.replace(" ", "_")
             .str.replace("-", "_")
)

In [24]:
mta["borough"].unique()

array(['M', 'SI', 'Bk', 'Q', 'Bx'], dtype=object)

In [25]:
mta_bk = mta[mta["borough"] == "Bk"]
mta_bk.head()

,gtfs_stop_id,station_id,complex_id,division,line,stop_name,borough,cbd,daytime_routes,structure,gtfs_latitude,gtfs_longitude,north_direction_label,south_direction_label,ada,ada_northbound,ada_southbound,ada_notes,georeference
2,S01,139,627,BMT,Franklin Shuttle,Franklin Av,Bk,False,S,Elevated,40.680596,-73.955827,Last Stop,Prospect Park,1,1,1,NaN,POINT (-73.955827 40.680596)
3,254,349,349,IRT,Eastern Pky,Junius St,Bk,False,3,Elevated,40.663515,-73.902447,Manhattan,New Lots,0,0,0,NaN,POINT (-73.902447 40.663515)
8,M11,97,97,BMT,Jamaica,Myrtle Av,Bk,False,M J Z,Elevated,40.697207,-73.935657,Outbound,Manhattan,0,0,0,NaN,POINT (-73.935657 40.697207)
10,F33,248,248,IND,6th Av - Culver,Avenue N,Bk,False,F,Elevated,40.615140,-73.974197,Manhattan,Southbound,0,0,0,NaN,POINT (-73.974197 40.61514)
11,233,336,336,IRT,Eastern Pky,Hoyt St,Bk,False,2 3,Subway,40.690545,-73.985065,Manhattan,Outbound,2,0,1,Outbound only,POINT (-73.985065 40.690545)


In [26]:
mta_bk_reduced = mta_bk[["stop_name", "gtfs_latitude", "gtfs_longitude", 
                         "daytime_routes", "line", "gtfs_stop_id", "complex_id"]]

mta_bk_reduced.head()

,stop_name,gtfs_latitude,gtfs_longitude,daytime_routes,line,gtfs_stop_id,complex_id
2,Franklin Av,40.680596,-73.955827,S,Franklin Shuttle,S01,627
3,Junius St,40.663515,-73.902447,3,Eastern Pky,254,349
8,Myrtle Av,40.697207,-73.935657,M J Z,Jamaica,M11,97
10,Avenue N,40.615140,-73.974197,F,6th Av - Culver,F33,248
11,Hoyt St,40.690545,-73.985065,2 3,Eastern Pky,233,336


In [27]:
mta_bk_reduced.shape

(169, 7)

In [28]:
mta_bk_reduced = mta_bk_reduced.dropna(subset=["gtfs_latitude", "gtfs_longitude"]).copy()
mta_bk_reduced.shape

(169, 7)

##### Construct Borough–Block–Lot (BBL)

A standardized BBL identifier is created to enable merging with PLUTO geographic data.

This allows:
- Latitude and longitude attachment
- Spatial feature engineering downstream

In [29]:
# Find nearest station to each property
from sklearn.neighbors import BallTree

In [30]:
# Create radians
resid_rad = np.radians(resid_with_geo_no_na[["latitude", "longitude"]])
mta_bk_rad = np.radians(mta_bk_reduced[["gtfs_latitude", "gtfs_longitude"]])

In [31]:
# Build BallTree
tree = BallTree(mta_bk_rad, metric="haversine")

In [32]:
# Query for nearest station
distances, indices = tree.query(resid_rad, k=1)

In [33]:
# Earth radius in miles
earth_radius = 3958.8

# Convert distances from radians to miles
distance_mi = distances * earth_radius

In [34]:
# Add results back to 'resid_with_geo_no_na'
resid_with_geo_no_na["nearest_station"] = indices.flatten()
resid_with_geo_no_na["distance_to_station"] = distance_mi.flatten()

resid_with_geo_no_na.head()

,borough_x,neighborhood,building_class_category,tax_class_at_present,block,lot,building_class_at_present,address,apartment_number,zip_code,residential_units,commercial_units,total_units,land_square_feet,gross_square_feet,year_built,tax_class_at_time_of_sale,building_class_at_time_of_sale,sale_price,sale_date,build_age_yrs,BBL,borough_y,latitude,longitude,nearest_station,distance_to_station
0,3,BATH BEACH,01 ONE FAMILY DWELLINGS,1,6364,72,A5,68 BAY 14TH STREET,NaN,11214,1,0,1,1950,972,1950,1,A5,860000,2025-11-24,76,3063640072,BK,40.607931,-74.007289,153,0.291308
1,3,BATH BEACH,01 ONE FAMILY DWELLINGS,1,6372,39,A9,86-29 19 AVENUE,NaN,11214,1,0,1,2417,1944,1930,1,A9,1180000,2025-11-19,96,3063720039,BK,40.605178,-74.000846,108,0.146919
2,3,BATH BEACH,01 ONE FAMILY DWELLINGS,1,6401,46,A1,144 BAY 17 STREET,NaN,11214,1,0,1,4833,2164,1925,1,A1,1300000,2025-01-13,101,3064010046,BK,40.604878,-74.006706,153,0.336323
3,3,BATH BEACH,01 ONE FAMILY DWELLINGS,1,6431,15,A9,221 BAY 13 STREET,NaN,11214,1,0,1,1530,2484,1920,1,A9,900000,2025-10-20,106,3064310015,BK,40.605037,-74.010865,153,0.519558
4,3,BATH BEACH,01 ONE FAMILY DWELLINGS,1,6431,56,A5,200 BAY 14 STREET,NaN,11214,1,0,1,2086,1458,1910,1,A5,930000,2025-02-05,116,3064310056,BK,40.605227,-74.010127,153,0.478792


#### Preprocessing Summary

At this stage, the dataset:

- Contains only valid residential transactions
- Has consistent structural variables
- Excludes extreme anomalies
- Is ready for feature engineering (geospatial distance, transformations, etc.)

The cleaned dataset is passed to the feature engineering notebook.

In [36]:
resid_with_geo_no_na.to_parquet(CLEAN_DATA_DIR / "01_preprocessing.parquet", index=False)